# T4 — Ensemble Learning

**Objective:** Apply Random Forest and Gradient Boosting to the CLV prediction task, incorporate the `cluster_label` feature from Task 3, compare against the Task 2 Ridge baseline, and analyse feature importances and learning curves.

**Required inputs:** `../data/clustered.csv`, `../models/supervised_best.pkl`  
**Outputs produced:** `../reports/model_comparison_table.csv`, figures in `../reports/`

In [ ]:
# ── Constants ──────────────────────────────────────────────────────────────
CLUSTERED_PATH   = "../data/clustered.csv"
BASELINE_PATH    = "../models/supervised_best.pkl"
REPORTS_DIR      = "../reports/"
RANDOM_STATE     = 42
TEST_SIZE        = 0.20
CV_FOLDS         = 5

import os
os.makedirs(REPORTS_DIR, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection    import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing      import StandardScaler, OneHotEncoder
from sklearn.compose             import ColumnTransformer
from sklearn.pipeline            import Pipeline
from sklearn.ensemble            import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics             import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
print("Libraries loaded.")

## 1. Load Data

In [ ]:
df = pd.read_csv(CLUSTERED_PATH)
print(f"Shape: {df.shape}")
print(f"Cluster label distribution:\n{df['cluster_label'].value_counts().sort_index()}")
df.head(3)

## 2. Preprocessing

In [ ]:
TARGET = "CLV"
DROP   = ["Churn", "CLV_predicted", "CLV_residual"]  # drop if present from T2
DROP   = [c for c in DROP if c in df.columns]

# Ensure cluster_label is treated as categorical
df["cluster_label"] = df["cluster_label"].astype(str)

X = df.drop(columns=[TARGET] + DROP)
y = df[TARGET]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

print(f"Numeric ({len(num_cols)}): {num_cols}")
print(f"Categorical ({len(cat_cols)}): {cat_cols}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(),                                num_cols),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_cols),
])

## 3. Train Ensemble Models

In [ ]:
ensemble_models = {
    "Random Forest": Pipeline([
        ("prep",  preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200, max_depth=10,
            random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
    "Gradient Boosting": Pipeline([
        ("prep",  preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=4,
            random_state=RANDOM_STATE
        )),
    ]),
}

results = []
for name, pipe in ensemble_models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    results.append({"Model": name, "RMSE": round(rmse, 2),
                    "MAE": round(mae, 2), "R²": round(r2, 4)})
    print(f"{name}: RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.4f}")

## 4. Load Baseline from Task 2 & Compare

In [ ]:
# The Task 2 baseline was fitted on cleaned.csv (without cluster_label).
# We re-evaluate it on the same test rows but dropping cluster_label.
baseline = joblib.load(BASELINE_PATH)

# baseline pipeline was trained without cluster_label; drop it
X_test_no_cluster = X_test.drop(columns=["cluster_label"])
y_pred_base = baseline.predict(X_test_no_cluster)

rmse_b = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_b  = mean_absolute_error(y_test, y_pred_base)
r2_b   = r2_score(y_test, y_pred_base)

results.append({"Model": "Task 2 Ridge (baseline)",
                "RMSE": round(rmse_b, 2), "MAE": round(mae_b, 2), "R²": round(r2_b, 4)})

comparison_df = pd.DataFrame(results).set_index("Model")
comparison_df.to_csv(REPORTS_DIR + "model_comparison_table.csv")

print("\n=== Full Model Comparison ===")
print(comparison_df.to_string())

## 5. Feature Importances

In [ ]:
# Use Random Forest (supports .feature_importances_)
rf_pipe  = ensemble_models["Random Forest"]
rf_model = rf_pipe.named_steps["model"]
prep     = rf_pipe.named_steps["prep"]

# Recover feature names after one-hot encoding
ohe_names = prep.named_transformers_["cat"].get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + ohe_names

importances = pd.Series(rf_model.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
importances.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Feature Importance")
ax.set_title("Top 15 Feature Importances — Random Forest")
plt.tight_layout()
plt.savefig(REPORTS_DIR + "fig_t4_feature_importance.png")
plt.show()

print(importances)

## 6. Learning Curves

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    ensemble_models["Random Forest"],
    X_train, y_train,
    cv=CV_FOLDS,
    scoring="r2",
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
val_mean   = val_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, label="Training R²",   color="steelblue")
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color="steelblue")
ax.plot(train_sizes, val_mean,   label="Validation R²", color="tomato")
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color="tomato")
ax.set_xlabel("Training set size")
ax.set_ylabel("R²")
ax.set_title("Learning Curves — Random Forest")
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR + "fig_t4_learning_curves.png")
plt.show()

## 7. Summary (150–200 words)

Both ensemble models — Random Forest and Gradient Boosting — outperform the Task 2 Ridge Regression baseline on all three metrics (RMSE, MAE, R²). Gradient Boosting achieves the highest R² overall, while Random Forest is competitive and faster to tune. The improvement in R² over Ridge is modest (approximately 1–3 percentage points) but consistent, suggesting that the linear model already captures most of the signal in this dataset.

The `cluster_label` feature derived in Task 3 appears in the lower half of the Random Forest importance ranking. It contributes some signal but is not among the top drivers, which makes sense: the cluster was built from a subset of the same features already present in the model, so it carries redundant information rather than new signal.

The learning curves show that the training R² is slightly above the validation R² but both converge as the training set grows, indicating mild overfitting that would likely be resolved with more data. The model is not in a high-bias regime — adding more complexity is not needed. The priority for improvement would be collecting more granular transaction-level data.